# Week 03 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 03 Day 01: Vectors, matrices, and transformations

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build matrix and optimization intuition while strengthening practical data handling skills.

## Continuity and Handoff
- Previous checkpoint: Week 02 Day 07: Portfolio Project
- Previous lesson file: content/week-02/day-07.md
- Today's deliverable: Implement matrix operations and shape checks using NumPy.
- Next handoff target: Week 03 Day 02: Eigenvalues, eigenvectors, and PCA intuition
- Next lesson file: content/week-03/day-02.md

## Theory Concepts

### Concept 1: Vector space intuition
Vector space intuition is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Matrix multiplication in factor exposure
Matrix multiplication in factor exposure is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Linear transformations for feature pipelines
Linear transformations for feature pipelines is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Map asset returns through a synthetic factor-loading matrix.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Vectors, matrices, and transformations'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $102.00 to $103.22.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (103.224-102.000)/102.000 = 0.012000.
Final answer: Simple return = 1.20%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0112.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0112 = 0.1778.
Final answer: Annualized volatility = 17.78%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 17.78%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1778 = 0.6187.
Final answer: Sharpe ratio = 0.62.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement matrix operations and shape checks using NumPy.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 03 Day 01, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Vectors, matrices, and transformations' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Vectors, matrices, and transformations as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Why do silent shape mismatches cause costly bugs?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 03 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(301)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_48764/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019175227834993,
 'annualized_return': 0.14223481807931715,
 'annualized_volatility': 0.19307143164265209,
 'max_drawdown': -0.3371726482138069,
 'spy_rate_corr': 0.008052785949585447}

## Week 03 Day 01 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [3]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 104.000
price_t = 104.884
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.008500)
print('  gross_return_expected  =', 1.008500)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 104.0
  P_t    : 104.884
  r_t    : 0.0085 => 0.85%
  1+r_t  : 1.0085

Expected numeric check:
  simple_return_expected = 0.0085
  gross_return_expected  = 1.0085

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 03 Day 02: Eigenvalues, eigenvectors, and PCA intuition

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build matrix and optimization intuition while strengthening practical data handling skills.

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 01: Vectors, matrices, and transformations
- Previous lesson file: content/week-03/day-01.md
- Today's deliverable: Build PCA diagnostic plots and explained-variance table.
- Next handoff target: Week 03 Day 03: Optimization fundamentals
- Next lesson file: content/week-03/day-03.md

## Theory Concepts

### Concept 1: Variance directions in data
Variance directions in data is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Principal components as latent factors
Principal components as latent factors is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Dimensionality reduction tradeoffs
Dimensionality reduction tradeoffs is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Apply PCA to correlated return features and interpret first components.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Eigenvalues, eigenvectors, and PCA intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $104.00 to $105.35.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (105.352-104.000)/104.000 = 0.013000.
Final answer: Simple return = 1.30%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0119.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0119 = 0.1889.
Final answer: Annualized volatility = 18.89%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 18.89%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1889 = 0.5823.
Final answer: Sharpe ratio = 0.58.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build PCA diagnostic plots and explained-variance table.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 03 Day 02, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Eigenvalues, eigenvectors, and PCA intuition' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Eigenvalues, eigenvectors, and PCA intuition as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
When can PCA remove useful signal?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 03 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [4]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(302)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_48764/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019169418923273,
 'annualized_return': 0.1422347913392359,
 'annualized_volatility': 0.19307146686187648,
 'max_drawdown': -0.33717266542154367,
 'spy_rate_corr': 0.008052798312080552}

## Week 03 Day 02 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [5]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 105.000
price_t = 105.945
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 105.0
  P_t    : 105.945
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Expected numeric check:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 03 Day 03: Optimization fundamentals

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build matrix and optimization intuition while strengthening practical data handling skills.

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 02: Eigenvalues, eigenvectors, and PCA intuition
- Previous lesson file: content/week-03/day-02.md
- Today's deliverable: Code constrained optimization with objective logging.
- Next handoff target: Week 03 Day 04: Pandas data engineering workflow
- Next lesson file: content/week-03/day-04.md

## Theory Concepts

### Concept 1: Objective functions and constraints
Objective functions and constraints is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Constrained optimization in portfolios
Constrained optimization in portfolios is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Regularization and numerical stability
Regularization and numerical stability is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Solve a constrained minimum-variance allocation toy problem.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Optimization fundamentals'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $106.00 to $107.48.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (107.484-106.000)/106.000 = 0.014000.
Final answer: Simple return = 1.40%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0126.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0126 = 0.2000.
Final answer: Annualized volatility = 20.00%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 20.00%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2000 = 0.5499.
Final answer: Sharpe ratio = 0.55.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Code constrained optimization with objective logging.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 03 Day 03, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Optimization fundamentals' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Optimization fundamentals as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
What assumptions are hidden inside quadratic optimization?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 03 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [6]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(303)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_48764/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019169418923273,
 'annualized_return': 0.1422347913392359,
 'annualized_volatility': 0.19307146686187648,
 'max_drawdown': -0.33717266542154367,
 'spy_rate_corr': 0.008052798312080552}

## Week 03 Day 03 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [7]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 106.000
price_t = 107.007
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009500)
print('  gross_return_expected  =', 1.009500)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 106.0
  P_t    : 107.007
  r_t    : 0.0095 => 0.95%
  1+r_t  : 1.0095

Expected numeric check:
  simple_return_expected = 0.0095
  gross_return_expected  = 1.0095

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 03 Day 04: Pandas data engineering workflow

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build matrix and optimization intuition while strengthening practical data handling skills.

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 03: Optimization fundamentals
- Previous lesson file: content/week-03/day-03.md
- Today's deliverable: Create a reusable data cleaning pipeline function.
- Next handoff target: Week 03 Day 05: SQL basics for quant datasets
- Next lesson file: content/week-03/day-05.md

## Theory Concepts

### Concept 1: Index alignment and time-aware joins
Index alignment and time-aware joins is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Groupby/rolling windows
Groupby/rolling windows is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Missing data treatment strategies
Missing data treatment strategies is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Align multi-asset price data with macro feature releases.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Pandas data engineering workflow'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $108.00 to $109.62.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (109.620-108.000)/108.000 = 0.015000.
Final answer: Simple return = 1.50%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0133.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0133 = 0.2111.
Final answer: Annualized volatility = 21.11%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 21.11%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2111 = 0.5210.
Final answer: Sharpe ratio = 0.52.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a reusable data cleaning pipeline function.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 03 Day 04, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Pandas data engineering workflow' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Pandas data engineering workflow as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
How can forward-fill cause subtle leakage?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 03 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(304)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_48764/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916941892327,
 'annualized_return': 0.1422347913392359,
 'annualized_volatility': 0.19307139757375744,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052804393332376}

## Week 03 Day 04 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [9]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 107.000
price_t = 108.070
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 107.0
  P_t    : 108.07
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Expected numeric check:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 03 Day 05: SQL basics for quant datasets

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build matrix and optimization intuition while strengthening practical data handling skills.

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 04: Pandas data engineering workflow
- Previous lesson file: content/week-03/day-04.md
- Today's deliverable: Write SQL queries for instrument-level and portfolio-level summaries.
- Next handoff target: Week 03 Day 06: Revision Sprint
- Next lesson file: content/week-03/day-06.md

## Theory Concepts

### Concept 1: Relational schema design
Relational schema design is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Filtering, aggregation, and joins
Filtering, aggregation, and joins is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Time-series query patterns
Time-series query patterns is a core part of 'Linear algebra, optimization, and data engineering'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Query daily returns and aggregate monthly stats from a SQLite table.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'SQL basics for quant datasets'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $110.00 to $111.76.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (111.760-110.000)/110.000 = 0.016000.
Final answer: Simple return = 1.60%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0140.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0140 = 0.2222.
Final answer: Annualized volatility = 22.22%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 22.22%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2222 = 0.4950.
Final answer: Sharpe ratio = 0.49.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Write SQL queries for instrument-level and portfolio-level summaries.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 03 Day 05, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'SQL basics for quant datasets' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through SQL basics for quant datasets as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
What makes a schema future-proof for research iteration?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 03 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [10]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(305)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_48764/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.0019175227834993,
 'annualized_return': 0.14223481807931715,
 'annualized_volatility': 0.19307143164265209,
 'max_drawdown': -0.3371726482138069,
 'spy_rate_corr': 0.008052785949585447}

## Week 03 Day 05 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [11]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 108.000
price_t = 109.134
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010500)
print('  gross_return_expected  =', 1.010500)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 108.0
  P_t    : 109.134
  r_t    : 0.0105 => 1.05%
  1+r_t  : 1.0105

Expected numeric check:
  simple_return_expected = 0.0105
  gross_return_expected  = 1.0105

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 03 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 05: SQL basics for quant datasets
- Previous lesson file: content/week-03/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 03 Day 07: Portfolio Project
- Next lesson file: content/week-03/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Rework one optimization example by hand
- Recreate one pandas pipeline from memory
- Write and test 3 SQL queries without references

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 03 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 06: Revision Sprint
- Previous lesson file: content/week-03/day-06.md
- Today's deliverable: Market data pipeline prototype
- Next handoff target: Week 04 Day 01: Return decomposition and compounding effects
- Next lesson file: content/week-04/day-01.md

## Project Title
Market data pipeline prototype

## Problem Statement
Build a robust ETL workflow that ingests, cleans, aligns, and stores market data.

## Data Sources
- Yahoo Finance
- FRED
- Local SQLite storage

## Implementation Steps
1. Design schema for prices and features
2. Ingest and normalize data
3. Handle missing values and alignment
4. Store transformed outputs
5. Validate data quality with checks

## Evaluation Metrics
- Data completeness
- Alignment errors
- Pipeline runtime
- Validation pass rate

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
